# Using an AI assistant for SQL, with checks

Practice notebook for the datafromzero SQL lesson **Using an AI assistant for SQL, with checks**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

The drafts in this notebook were written by hand for the lesson. They are not the output of a real AI product.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Write the prompt: schema, question, output grain

> I use DuckDB. Tables: orders(order_id, customer_id, store_id, ordered_at, status, payment_method); order_items(order_item_id, order_id, product_id, quantity, discount_usd); products(product_id, product_name, category, unit_price_usd); stores(store_id, store_name, city, opened_on). One row of orders is one order; one row of order_items is one product line inside an order. The amount of a line is quantity × unit_price_usd − discount_usd. status is 'completed' or 'canceled'. Question: the average order value of completed orders for each store. Output: one row per store with store_name, orders and avg_order_value. Give only the SQL.

It holds column names and no customer data. Never paste real customers into an assistant.

### Step 2. Read the draft clause by clause, then run it

The draft counts with `COUNT(*)` after three joins. What does one row of the join stand for? Expect 3 rows and 3 columns.

In [ ]:
con.sql("""
SELECT s.store_name,
       COUNT(*) AS orders,
       ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd) / COUNT(*), 2) AS avg_order_value
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
JOIN stores AS s ON s.store_id = o.store_id
WHERE o.status = 'completed'
GROUP BY s.store_name
ORDER BY s.store_name
""").show()

### Step 3. Reconcile the count

Add up the draft's `orders` column and compare it with this count.

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders FROM orders WHERE status = 'completed'
""").show()

### Step 4. Known answer, by hand first

In the frame (orders 1458 to 1464), work out the completed Old Town and Campus averages on paper before you run the next cell.

In [ ]:
con.sql("""
SELECT s.store_name,
       COUNT(*) AS orders,
       ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd) / COUNT(*), 2) AS avg_order_value
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
JOIN stores AS s ON s.store_id = o.store_id
WHERE o.status = 'completed'
  AND o.order_id BETWEEN 1458 AND 1464
GROUP BY s.store_name
ORDER BY s.store_name
""").show()

### Step 5. Fix it and verify again

In [ ]:
con.sql("""
SELECT s.store_name,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
JOIN stores AS s ON s.store_id = o.store_id
WHERE o.status = 'completed'
GROUP BY s.store_name
ORDER BY s.store_name
""").show()

In [ ]:
con.sql("""
SELECT s.store_name,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
JOIN stores AS s ON s.store_id = o.store_id
WHERE o.status = 'completed'
  AND o.order_id BETWEEN 1458 AND 1464
GROUP BY s.store_name
ORDER BY s.store_name
""").show()

In [ ]:
con.sql("""
SELECT store_id, ROUND(SUM(quantity * unit_price_usd - discount_usd), 2) AS sales
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
WHERE o.status = 'completed'
GROUP BY store_id
ORDER BY store_id
""").show()

## Exercise 1. Customers who never ordered

The README says five customers signed up and never ordered. Run the draft, compare its row count with five, find out why with one query on `orders`, and write a draft that passes.

In [ ]:
con.sql("""
SELECT customer_id
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders)
""").show()

In [ ]:
# One query on orders that finds out why:

In [ ]:
# Your draft that passes:

## Exercise 2. The running total of completed sales, one row per day

Before you run anything, write one sentence in the first cell: how many rows the result should have and what its last value should be. Then run the draft (second cell), check it, and write a fixed query. Also write the prompt that states the output grain.

In [ ]:
# My expected result, in words:
#

In [ ]:
con.sql("""
SELECT CAST(o.ordered_at AS DATE) AS order_day,
       SUM(i.quantity * p.unit_price_usd - i.discount_usd) OVER (ORDER BY CAST(o.ordered_at AS DATE)) AS running_sales
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
WHERE o.status = 'completed'
ORDER BY order_day
""").show()

In [ ]:
# Prompt with the output grain:
#

In [ ]:
# Your fixed query:

## Exercise 3. Spot the flaw

A colleague checks that this draft returns 3 rows and that Harbor's 4,606.25 matches the pivot, and accepts it. What went wrong, and which check would have caught it?

In [ ]:
con.sql("""
SELECT s.store_name,
       ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd), 2) AS sales
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN products AS p ON p.product_id = i.product_id
JOIN stores AS s ON s.store_id = o.store_id
GROUP BY s.store_name
ORDER BY s.store_name
""").show()

In [ ]:
# Your checks and your fix: